# TP3 — Paso 1: cargar y revisar los libros

## Paso 1 Cargar y revisar los libros

Se conserva slug como identificador, title para mostrar el libro, authors y synopsis como datos de entrada. Solo se normalizan los espacios; se mantienen los demás caracteres porque pueden ser evidencia en la auditoría del Paso 2.

In [1]:
%pip install -q python-Levenshtein

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import unicodedata
from collections import defaultdict
from itertools import combinations

import Levenshtein

def normalizar_espacios(texto):
    return re.sub(r"\s+", " ", str(texto or "")).strip()

def normalizar_titulo(texto):
    texto = unicodedata.normalize("NFKC", str(texto or "")).casefold()
    texto = re.sub(r"[^\w\s]", " ", texto)
    return normalizar_espacios(texto)

def jaccard_palabras(texto_a, texto_b):
    palabras_a = set(re.findall(r"\w+", str(texto_a or "").casefold()))
    palabras_b = set(re.findall(r"\w+", str(texto_b or "").casefold()))
    union = palabras_a | palabras_b
    if not union:
        return 0.0
    return len(palabras_a & palabras_b) / len(union)

def limpiar_catalogo(datos):
    catalogo = datos.copy()
    catalogo["synopsis"] = catalogo["synopsis"].fillna("").apply(normalizar_espacios)
    conteos = {"sinopsis_vacia": 0, "slug_duplicado": 0, "sinopsis_duplicada": 0}

    vacias = catalogo["synopsis"].eq("")
    conteos["sinopsis_vacia"] = int(vacias.sum())
    catalogo = catalogo.loc[~vacias].copy()

    duplicados_slug = catalogo["slug"].duplicated()
    conteos["slug_duplicado"] = int(duplicados_slug.sum())
    catalogo = catalogo.loc[~duplicados_slug].copy()

    duplicadas = catalogo["synopsis"].duplicated()
    conteos["sinopsis_duplicada"] = int(duplicadas.sum())
    catalogo = catalogo.loc[~duplicadas].copy()

    return catalogo.reset_index(drop=True), conteos

UMBRAL_TITULO = 0.80
UMBRAL_SINOPSIS = 0.75
AUTORES_GENERICOS = {"varios autores", "anónimo", "anonimo"}

def generar_candidatos(datos):
    filas_por_autor = defaultdict(set)
    nombre_autor = {}
    for indice, autores in datos["authors"].items():
        if isinstance(autores, str):
            autores = [autores]
        elif not isinstance(autores, (list, tuple, set)):
            autores = []
        for autor in autores:
            autor = normalizar_espacios(autor)
            clave = autor.casefold()
            if clave and clave not in AUTORES_GENERICOS:
                filas_por_autor[clave].add(indice)
                nombre_autor[clave] = autor

    autores_compartidos = defaultdict(set)
    for clave, indices in filas_por_autor.items():
        for indice_a, indice_b in combinations(sorted(indices), 2):
            autores_compartidos[(indice_a, indice_b)].add(nombre_autor[clave])

    resultados = []
    for (indice_a, indice_b), autores in autores_compartidos.items():
        libro_a = datos.loc[indice_a]
        libro_b = datos.loc[indice_b]
        similitud_titulo = Levenshtein.ratio(
            normalizar_titulo(libro_a["title"]),
            normalizar_titulo(libro_b["title"]),
        )
        if similitud_titulo < UMBRAL_TITULO:
            continue
        similitud_sinopsis = jaccard_palabras(libro_a["synopsis"], libro_b["synopsis"])
        if similitud_sinopsis < UMBRAL_SINOPSIS:
            continue
        resultados.append({
            "slug_a": libro_a["slug"],
            "titulo_a": libro_a["title"],
            "slug_b": libro_b["slug"],
            "titulo_b": libro_b["title"],
            "autores_compartidos": ", ".join(sorted(autores)),
            "similitud_titulo": round(similitud_titulo, 3),
            "jaccard_sinopsis": round(similitud_sinopsis, 3),
        })

    columnas = ["slug_a", "titulo_a", "slug_b", "titulo_b", "autores_compartidos", "similitud_titulo", "jaccard_sinopsis"]
    if not resultados:
        return pd.DataFrame(columns=columnas)
    return pd.DataFrame(resultados, columns=columnas).sort_values(
        ["jaccard_sinopsis", "similitud_titulo"], ascending=False
    ).reset_index(drop=True)

### Cargar el catálogo

Los archivos de entrada están versionados en la carpeta data del repositorio. La celda los busca al ejecutar Jupyter desde la raíz o desde notebooks. En Colab, se pueden cargar los archivos en data o en el directorio actual.

In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import display

rutas_datos = [Path("data"), Path("../data"), Path("../../TP3"), Path("../../../TP3"), Path(".")]
directorio_datos = next(
    (ruta for ruta in rutas_datos if (ruta / "catalogo_tp3.jsonl").is_file()),
    None,
)
if directorio_datos is None:
    raise FileNotFoundError("No se encontró catalogo_tp3.jsonl en las rutas de datos probadas.")

catalogo_original = pd.read_json(directorio_datos / "catalogo_tp3.jsonl", lines=True)
columnas_paso1 = ["slug", "title", "authors", "synopsis"]
catalogo = catalogo_original[columnas_paso1].copy()
print(f"Registros originales: {len(catalogo):,}")

Registros originales: 8,747


### Revisar sinopsis repetidas

Primero se normalizan solo los espacios y se muestran los grupos con la misma sinopsis antes de quitar duplicados. Para esta práctica se conserva el primer registro de cada grupo, tal como indica el enunciado.

In [4]:
catalogo_revision = catalogo.copy()
catalogo_revision["synopsis"] = catalogo_revision["synopsis"].fillna("").apply(normalizar_espacios)
con_sinopsis = catalogo_revision.loc[catalogo_revision["synopsis"].ne("")]
duplicadas = con_sinopsis.loc[con_sinopsis["synopsis"].duplicated(keep=False)]

grupos_sinopsis = (
    duplicadas.groupby("synopsis", as_index=False)
    .agg(
        cantidad=("slug", "size"),
        titulos=("title", lambda titulos: " / ".join(titulos.iloc[:5]) + (" / …" if len(titulos) > 5 else "")),
    )
)
grupos_sinopsis["fragmento"] = grupos_sinopsis["synopsis"].str.slice(0, 140)
filas_repetidas = int((grupos_sinopsis["cantidad"] - 1).sum())
print(f"Grupos con sinopsis repetida: {len(grupos_sinopsis)}")
print(f"Registros que se quitarían por este motivo: {filas_repetidas}")
display(grupos_sinopsis.drop(columns="synopsis").sort_values("cantidad", ascending=False).reset_index(drop=True))

Grupos con sinopsis repetida: 33
Registros que se quitarían por este motivo: 98


,cantidad,titulos,fragmento
0,58,Alemán – Unidad 40 / Alemán – Unidad 15 / Alem...,El Curso de Alemán consta de 96 Unidades didác...
1,7,Nueva Dimensión 7 / Nueva Dimensión 5 / Nueva ...,Revista española de ciencia ficción y fantasía...
2,4,Teatro crítico universal VII / Teatro crítico ...,El «Teatro crítico universal» (teatro ha de en...
3,3,Dionisíacas Cantos XXV-XXXVI / Dionisíacas Can...,"Nono (siglo V d. C.), natural de Panópolis, en..."
4,3,El fin de una cuadrilla / El santón de la mont...,"El autor, llamado Kara Ben Nemsi (Carlos, hijo..."
5,2,Prácticas sexuales alternativas (I) / Práctica...,Al repasar la larga lista de prácticas sexuale...
6,2,Los años irreverentes. Escritos completos en E...,"Como todas las cosas que suceden en la vida, h..."
7,2,Comunicaciones unificadas con Elastix / Comuni...,Comunicaciones Unificadas con Elastix es una o...
8,2,La Historia Interminable – Color / La Historia...,"Bastián Baltasar Bux, es un niño tímido al que..."
9,2,El 19 de marzo y el 2 de mayo / Bailén,El gran friso narrativo de los Episodios Nacio...


### Limpiar registros y conservar una sinopsis por grupo

In [5]:
catalogo_limpio, conteos_eliminacion = limpiar_catalogo(catalogo)

resumen_limpieza = pd.DataFrame(
    [{"motivo": motivo, "eliminados": cantidad} for motivo, cantidad in conteos_eliminacion.items()]
)
resumen_limpieza.loc[len(resumen_limpieza)] = ["disponibles tras limpieza exacta", len(catalogo_limpio)]
display(resumen_limpieza)
print(f"Libros disponibles tras eliminar sinopsis vacías y repetidas: {len(catalogo_limpio):,}")


,motivo,eliminados
0,sinopsis_vacia,1
1,slug_duplicado,0
2,sinopsis_duplicada,98
3,disponibles tras limpieza exacta,8648


Libros disponibles tras eliminar sinopsis vacías y repetidas: 8,648


### Revisar títulos y sinopsis parecidos

Los pares se numeran para revisar cada caso. Los umbrales solo proponen candidatos; no determinan fusiones.


In [6]:
candidatos = generar_candidatos(catalogo_limpio)
candidatos.insert(0, "id_candidato", range(1, len(candidatos) + 1))
print(f"Pares que superan ambos filtros: {len(candidatos)}")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None, "display.width", 220):
    display(candidatos)


Pares que superan ambos filtros: 66


,id_candidato,slug_a,titulo_a,slug_b,titulo_b,autores_compartidos,similitud_titulo,jaccard_sinopsis
0,1,comentarios-psicologicos-sobre-las-ensenanzas-de-gurdjieff-y-ouspensky-2,Comentarios psicológicos sobre las enseñanzas de Gurdjieff y Ouspensky,comentarios-psicologicos-sobre-las-ensenanzas-de-gurdjieff-y-ouspensky,Comentarios psicológicos sobre las enseñanzas de Gurdjieff y Ouspensky,Maurice Nicoll,1.000,1.000
1,2,vidas-magicas-e-inquisicion-i,Vidas mágicas e Inquisición I,vidas-magicas-e-inquisicion-ii,Vidas mágicas e Inquisición II,Julio Caro Baroja,0.983,1.000
2,3,orgullo-y-poco-juicio-parte-2,Orgullo y poco Juicio. Parte 2,orgullo-y-poco-juicio-parte-1,Orgullo y poco Juicio. Parte 1,Lena Valenti,0.966,1.000
3,4,teatro-critico-universal-vii,Teatro crítico universal VII,teatro-critico-universal-i,Teatro crítico universal I,Benito Jerónimo Feijoo,0.963,1.000
4,5,epigramas-ii,Epigramas II,epigramas-i,Epigramas I,Marco Valerio Marcial,0.957,1.000
5,6,cartas-eruditas-y-curiosas-ii,"Cartas eruditas, y curiosas II",cartas-eruditas-y-curiosas-v,"Cartas eruditas, y curiosas V",Benito Jerónimo Feijoo,0.947,1.000
6,7,manual-para-viajeros-por-andalucia-y-lectores-en-casa-granada,Manual para viajeros por Andalucía y lectores en casa. Granada,manual-para-viajeros-por-andalucia-y-lectores-en-casa,Manual para viajeros por Andalucía y lectores en casa,Richard Ford,0.930,1.000
7,8,ella-hija-de-la-sabiduria,"Ella, Hija de la Sabiduría",ella-hija-de-la-sabiduria-h-rider-haggard,Ella. Hija de la sabiduría,H. Rider Haggard,1.000,0.989
8,9,teatro-critico-universal-vii,Teatro crítico universal VII,teatro-critico-universal-ii,Teatro crítico universal II,Benito Jerónimo Feijoo,0.982,0.987
9,10,teatro-critico-universal-i,Teatro crítico universal I,teatro-critico-universal-ii,Teatro crítico universal II,Benito Jerónimo Feijoo,0.981,0.987


### Decisiones manuales y catálogo final

Tras revisar los candidatos, se aprobaron las fusiones #1, #7, #8, #11 y #31. En cada par se conserva la primera ficha según el orden del catálogo y se quita la segunda. Las razones aparecen junto con los títulos, el autor y los puntajes.

El candidato #7, «Manual para viajeros por Andalucía y lectores en casa. Granada» / «Manual para viajeros por Andalucía y lectores en casa», también se une. Comparten autor y su Jaccard de sinopsis es 1,00: tienen el mismo conjunto de palabras al convertirlas a minúsculas, aunque las sinopsis no sean necesariamente idénticas carácter por carácter. Se conserva la primera ficha del catálogo y se elimina la segunda, aunque su título no especifique Granada.

Los otros dos ejemplos de separación son «Conversaciones con Dios I/II» y «La escultura del siglo XIX/XX»: se parecen en el título, pero describen volúmenes o contenidos diferentes.


In [7]:
motivos_fusion = {
    1: "Mismo autor y título idéntico; las sinopsis difieren solo en detalles de puntuación.",
    7: "Mismo autor y el mismo conjunto de palabras en la sinopsis (Jaccard = 1,00); se conserva la primera ficha aunque la otra especifique Granada.",
    8: "Mismo autor y obra; el título cambia puntuación y mayúsculas, y las sinopsis son casi iguales.",
    11: "Mismo autor y obra; una ficha omite el artículo inicial «El» y las sinopsis casi coinciden.",
    31: "Mismo autor y obra; una ficha omite el artículo inicial «El» y las sinopsis describen el mismo libro.",
}
ids_fusion_aprobados = list(motivos_fusion)
fusiones_aprobadas = candidatos.loc[candidatos["id_candidato"].isin(ids_fusion_aprobados)].copy()
fusiones_aprobadas["decision"] = "unir"
fusiones_aprobadas["motivo"] = fusiones_aprobadas["id_candidato"].map(motivos_fusion)
fusiones_aprobadas["slug_conservado"] = fusiones_aprobadas["slug_a"]
fusiones_aprobadas["slug_eliminado"] = fusiones_aprobadas["slug_b"]
with pd.option_context("display.max_colwidth", None, "display.width", 220):
    display(fusiones_aprobadas[["id_candidato", "titulo_a", "titulo_b", "autores_compartidos", "similitud_titulo", "jaccard_sinopsis", "decision", "motivo", "slug_conservado", "slug_eliminado"]])

slugs_a_eliminar = fusiones_aprobadas["slug_eliminado"].tolist()
catalogo_final = catalogo_limpio.loc[~catalogo_limpio["slug"].isin(slugs_a_eliminar)].reset_index(drop=True)

pares_separados = [
    ("conversaciones-con-dios-i", "conversaciones-con-dios-ii", "Son volúmenes distintos; sus sinopsis describen contenidos diferentes."),
    ("la-escultura-del-siglo-xix", "la-escultura-del-siglo-xx", "Se refieren a siglos y contenidos distintos."),
]
separaciones = []
for slug_a, slug_b, motivo in pares_separados:
    libro_a = catalogo_limpio.loc[catalogo_limpio["slug"].eq(slug_a)].iloc[0]
    libro_b = catalogo_limpio.loc[catalogo_limpio["slug"].eq(slug_b)].iloc[0]
    separaciones.append({
        "titulo_a": libro_a["title"],
        "titulo_b": libro_b["title"],
        "autor": ", ".join(sorted(set(libro_a["authors"]) & set(libro_b["authors"]))),
        "similitud_titulo": round(Levenshtein.ratio(normalizar_titulo(libro_a["title"]), normalizar_titulo(libro_b["title"])), 3),
        "jaccard_sinopsis": round(jaccard_palabras(libro_a["synopsis"], libro_b["synopsis"]), 3),
        "decision": "separar",
        "motivo": motivo,
    })
with pd.option_context("display.max_colwidth", None, "display.width", 220):
    display(pd.DataFrame(separaciones))

resumen_final = pd.DataFrame([
    {"motivo": motivo, "eliminados": cantidad}
    for motivo, cantidad in conteos_eliminacion.items()
])
resumen_final.loc[len(resumen_final)] = ["fusion aproximada aprobada", len(slugs_a_eliminar)]
resumen_final.loc[len(resumen_final)] = ["libros finales", len(catalogo_final)]
display(resumen_final)
print(f"Slugs únicos: {catalogo_final['slug'].nunique():,} de {len(catalogo_final):,}")
print(f"Sinopsis no vacías: {catalogo_final['synopsis'].ne('').sum():,}")
print(f"Sinopsis únicas: {catalogo_final['synopsis'].nunique():,} de {len(catalogo_final):,}")

muestra_final = catalogo_final[["slug", "title", "authors", "synopsis"]].head(5)
display(muestra_final)


,id_candidato,titulo_a,titulo_b,autores_compartidos,similitud_titulo,jaccard_sinopsis,decision,motivo,slug_conservado,slug_eliminado
0,1,Comentarios psicológicos sobre las enseñanzas de Gurdjieff y Ouspensky,Comentarios psicológicos sobre las enseñanzas de Gurdjieff y Ouspensky,Maurice Nicoll,1.000,1.000,unir,Mismo autor y título idéntico; las sinopsis difieren solo en detalles de puntuación.,comentarios-psicologicos-sobre-las-ensenanzas-de-gurdjieff-y-ouspensky-2,comentarios-psicologicos-sobre-las-ensenanzas-de-gurdjieff-y-ouspensky
6,7,Manual para viajeros por Andalucía y lectores en casa. Granada,Manual para viajeros por Andalucía y lectores en casa,Richard Ford,0.930,1.000,unir,"Mismo autor y el mismo conjunto de palabras en la sinopsis (Jaccard = 1,00); se conserva la primera ficha aunque la otra especifique Granada.",manual-para-viajeros-por-andalucia-y-lectores-en-casa-granada,manual-para-viajeros-por-andalucia-y-lectores-en-casa
7,8,"Ella, Hija de la Sabiduría",Ella. Hija de la sabiduría,H. Rider Haggard,1.000,0.989,unir,"Mismo autor y obra; el título cambia puntuación y mayúsculas, y las sinopsis son casi iguales.",ella-hija-de-la-sabiduria,ella-hija-de-la-sabiduria-h-rider-haggard
10,11,El príncipe y el mendigo,Príncipe y mendigo,Mark Twain,0.857,0.985,unir,Mismo autor y obra; una ficha omite el artículo inicial «El» y las sinopsis casi coinciden.,el-principe-y-el-mendigo,principe-y-mendigo
30,31,El contrato social,Contrato social,Jean-Jacques Rousseau,0.909,0.917,unir,Mismo autor y obra; una ficha omite el artículo inicial «El» y las sinopsis describen el mismo libro.,el-contrato-social,contrato-social


,titulo_a,titulo_b,autor,similitud_titulo,jaccard_sinopsis,decision,motivo
0,Conversaciones Con Dios I,Conversaciones Con Dios II,Neale Donald Walsch,0.98,0.094,separar,Son volúmenes distintos; sus sinopsis describen contenidos diferentes.
1,La escultura del siglo XIX,La escultura del siglo XX,Ernesto Ballesteros Arranz,0.98,0.130,separar,Se refieren a siglos y contenidos distintos.


,motivo,eliminados
0,sinopsis_vacia,1
1,slug_duplicado,0
2,sinopsis_duplicada,98
3,fusion aproximada aprobada,5
4,libros finales,8643


Slugs únicos: 8,643 de 8,643
Sinopsis no vacías: 8,643
Sinopsis únicas: 8,643 de 8,643


,slug,title,authors,synopsis
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],La Humanidad ha entrado en una nueva fase evol...
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],La extracción de ADN de los huesos de nuestros...
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],Teotihuacan es la primera gran ciudad del cent...
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],Entre los siglos VIII y VI a. C. los fenicios ...
4,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],Las raíces evolutivas del sufrimiento humano U...


## Paso 2 Auditar el catálogo

La muestra etiquetada se divide por libro antes de evaluar los detectores. La partición de desarrollo queda disponible para los subpasos siguientes y la partición de prueba se reserva para medir los resultados.

In [8]:
from sklearn.model_selection import train_test_split

muestra = pd.read_csv(directorio_datos / "muestra_etiquetada.csv", keep_default_na=False)
desarrollo, prueba = train_test_split(
    muestra,
    test_size=0.3,
    random_state=42,
    stratify=muestra["contaminado"],
)

resumen_particion = pd.DataFrame([
    {
        "particion": nombre,
        "libros": len(datos),
        "limpios": int(datos["contaminado"].eq(0).sum()),
        "contaminados": int(datos["contaminado"].eq(1).sum()),
    }
    for nombre, datos in [("desarrollo", desarrollo), ("prueba", prueba)]
])
display(resumen_particion)

,particion,libros,limpios,contaminados
0,desarrollo,350,280,70
1,prueba,150,120,30


### 2.1 Expresiones regulares

Se buscan por separado URLs, correos y teléfonos en la sinopsis original. La combinación marca el libro cuando encuentra al menos una de esas señales. Las formas disimuladas se revisarán en el apartado 2.2.

In [9]:
from sklearn.metrics import precision_score, recall_score

patrones_regex = [
    ("URL", "marca_url", re.compile(r"https?://|www\.", re.IGNORECASE)),
    ("Correo", "marca_correo", re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", re.IGNORECASE)),
    ("Teléfono", "marca_telefono", re.compile(r"\+\d(?:[\s().-]?\d){7,14}")),
]

for nombre, columna, patron in patrones_regex:
    prueba[columna] = prueba["synopsis"].str.contains(patron, na=False)

columnas_individuales = [columna for _, columna, _ in patrones_regex]
prueba["marca_combinada"] = prueba[columnas_individuales].any(axis=1)

columnas_detectores = {nombre: columna for nombre, columna, _ in patrones_regex}
columnas_detectores["Combinada"] = "marca_combinada"
y_prueba = prueba["contaminado"].astype(int)
resultados_regex = []
for nombre, columna in columnas_detectores.items():
    predicciones = prueba[columna].astype(int)
    resultados_regex.append({
        "detector": nombre,
        "precision": round(precision_score(y_prueba, predicciones, zero_division=0), 2),
        "recall": round(recall_score(y_prueba, predicciones, zero_division=0), 2),
        "libros_marcados": int(predicciones.sum()),
    })
tabla_metricas_regex = pd.DataFrame(resultados_regex)
display(tabla_metricas_regex)

falsos_positivos_regex = prueba.loc[
    prueba["contaminado"].eq(0) & prueba["marca_combinada"],
].copy()
falsos_positivos_regex["senales"] = falsos_positivos_regex.apply(
    lambda fila: ", ".join(
        nombre for nombre, columna, _ in patrones_regex if fila[columna]
    ),
    axis=1,
)
print(f"Falsos positivos de la combinación: {len(falsos_positivos_regex)}")
display(falsos_positivos_regex[["slug", "title", "senales", "synopsis"]])

if falsos_positivos_regex.empty:
    desarrollo_marcado = desarrollo.copy()
    for _, columna, patron in patrones_regex:
        desarrollo_marcado[columna] = desarrollo_marcado["synopsis"].str.contains(patron, na=False)
    desarrollo_marcado["marca_combinada"] = desarrollo_marcado[columnas_individuales].any(axis=1)
    falsos_positivos_desarrollo = desarrollo_marcado.loc[
        desarrollo_marcado["contaminado"].eq(0) & desarrollo_marcado["marca_combinada"],
    ].copy()
    falsos_positivos_desarrollo["senales"] = falsos_positivos_desarrollo.apply(
        lambda fila: ", ".join(
            nombre for nombre, columna, _ in patrones_regex if fila[columna]
        ),
        axis=1,
    )
    print("Ejemplos limpios de desarrollo; no forman parte de las métricas de prueba:")
    display(falsos_positivos_desarrollo[["slug", "title", "senales", "synopsis"]])


,detector,precision,recall,libros_marcados
0,URL,1.0,0.13,4
1,Correo,1.0,0.07,2
2,Teléfono,1.0,0.03,1
3,Combinada,1.0,0.23,7


Falsos positivos de la combinación: 0


,slug,title,senales,synopsis


Ejemplos limpios de desarrollo; no forman parte de las métricas de prueba:


,slug,title,senales,synopsis
234,que-pasaria-si,¿Que pasaría si…?,URL,Un libro en el que el humor y la ciencia se un...


### Respuesta a la pregunta del enunciado

**¿Qué tipo de contaminación detecta bien la regex y cuál no?**

Entre las señales buscadas, las URLs explícitas fueron las más detectadas: el patrón de URL marcó 4 de los 30 libros contaminados de prueba (recall global 0,13). El de correo marcó 2 (0,07) y el de teléfono 1 (0,03). La combinación OR marcó 7 de 30 (recall 0,23). Las URLs fueron la señal más útil en esta muestra, aunque el recall de la combinación sigue siendo bajo.

Las etiquetas solo indican si cada libro está contaminado; no identifican el tipo de contaminación. Por eso, esos valores son recall sobre los 30 libros contaminados en conjunto, no recall específico de libros con URL, correo o teléfono. Las regex tampoco detectan contaminación que no incluya esas señales, como enlaces disimulados o texto promocional sin datos de contacto.

La prueba no tuvo falsos positivos. En desarrollo, el libro limpio «¿Qué pasaría si…?» contiene el sitio real `http://www.xkcd.com` y lo marca la regex de URL. Ese caso se muestra aparte y no se usa para calcular métricas. Cada falso positivo puede quitar del catálogo un libro legítimo y dejar de recomendarse.

La muestra está enriquecida: contiene 20 % de libros contaminados, una proporción mayor que la del catálogo. Por eso, su precisión sirve para comparar estos detectores en la muestra, pero no representa directamente la precisión esperable al aplicarlos al catálogo completo.

### 2.2 Normalizar antes de buscar

La regex del apartado 2.1 busca señales explícitas. Algunos enlaces separan sus letras y signos, escriben los signos con palabras o usan caracteres Unicode parecidos a letras latinas. La función `normalizar(texto)` modifica una copia para buscar esas formas; las sinopsis originales se conservan.

Se mantienen los mismos patrones y la misma partición: desarrollo para revisar ejemplos y prueba para comparar las métricas.

In [10]:
homoglifos = str.maketrans({"е": "e", "а": "a"})


def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto or "")).lower()
    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
        and unicodedata.category(caracter) != "Cf"
    )
    texto = texto.translate(homoglifos)
    texto = re.sub(r"\[\s*\.\s*\]", ".", texto)
    texto = re.sub(r"\bpunto\b", ".", texto)
    texto = re.sub(r"\bbarra\b", "/", texto)
    texto = re.sub(
        r"\bh\s+t\s+t\s+p\s*s?\b",
        lambda coincidencia: re.sub(r"\s+", "", coincidencia.group()),
        texto,
    )
    texto = re.sub(r"\bw\s+w\s+w\b", "www", texto)
    texto = re.sub(r"\s*([:/@])\s*", r"\1", texto)
    texto = re.sub(r"(?<=\w)\s*\.\s*(?=\w)", ".", texto)
    return texto

In [11]:
prueba_2_2 = prueba.copy()
prueba_2_2["synopsis_normalizada"] = prueba_2_2["synopsis"].map(normalizar)

columnas_detectores_normalizados = {}
for nombre, columna, patron in patrones_regex:
    columna_normalizada = f"{columna}_normalizada"
    prueba_2_2[columna_normalizada] = prueba_2_2["synopsis_normalizada"].map(
        lambda texto: bool(patron.search(texto))
    )
    columnas_detectores_normalizados[nombre] = columna_normalizada

columnas_marcadas_normalizadas = list(columnas_detectores_normalizados.values())
prueba_2_2["marca_combinada_normalizada"] = prueba_2_2[
    columnas_marcadas_normalizadas
].any(axis=1)
columnas_detectores_normalizados["Combinada"] = "marca_combinada_normalizada"

y_prueba_2_2 = prueba_2_2["contaminado"].astype(int)
resultados_normalizados = []
for nombre, columna in columnas_detectores_normalizados.items():
    predicciones = prueba_2_2[columna].astype(int)
    resultados_normalizados.append({
        "detector": nombre,
        "precision": round(precision_score(y_prueba_2_2, predicciones, zero_division=0), 2),
        "recall": round(recall_score(y_prueba_2_2, predicciones, zero_division=0), 2),
        "libros_marcados": int(predicciones.sum()),
    })

tabla_metricas_regex_2_2 = pd.concat(
    [
        tabla_metricas_regex.assign(version="Original"),
        pd.DataFrame(resultados_normalizados).assign(version="Normalizada"),
    ],
    ignore_index=True,
)[["version", "detector", "precision", "recall", "libros_marcados"]]
display(tabla_metricas_regex_2_2)

falsos_positivos_normalizados = prueba_2_2.loc[
    prueba_2_2["contaminado"].eq(0)
    & prueba_2_2["marca_combinada_normalizada"]
].copy()
print(f"Falsos positivos de la combinación normalizada: {len(falsos_positivos_normalizados)}")
if falsos_positivos_normalizados.empty:
    print("La partición de prueba no tuvo falsos positivos normalizados.")
else:
    falsos_positivos_normalizados["senales"] = falsos_positivos_normalizados.apply(
        lambda fila: ", ".join(
            nombre
            for nombre, columna in columnas_detectores_normalizados.items()
            if nombre != "Combinada" and fila[columna]
        ),
        axis=1,
    )
    display(
        falsos_positivos_normalizados[
            ["slug", "title", "senales", "synopsis"]
        ]
    )

,version,detector,precision,recall,libros_marcados
0,Original,URL,1.0,0.13,4
1,Original,Correo,1.0,0.07,2
2,Original,Teléfono,1.0,0.03,1
3,Original,Combinada,1.0,0.23,7
4,Normalizada,URL,1.0,0.23,7
5,Normalizada,Correo,1.0,0.07,2
6,Normalizada,Teléfono,1.0,0.03,1
7,Normalizada,Combinada,1.0,0.33,10


Falsos positivos de la combinación normalizada: 0
La partición de prueba no tuvo falsos positivos normalizados.


La tabla compara cada regex y su combinación antes y después de normalizar, siempre sobre los mismos libros de prueba. Los ejemplos siguientes salen de desarrollo y no forman parte de las métricas.

In [12]:
desarrollo_2_2 = desarrollo.copy()
columnas_originales_desarrollo = []
columnas_normalizadas_desarrollo = []
for _, columna, patron in patrones_regex:
    columna_original = f"{columna}_original"
    columna_normalizada = f"{columna}_normalizada"
    desarrollo_2_2[columna_original] = desarrollo_2_2["synopsis"].map(
        lambda texto: bool(patron.search(texto))
    )
    desarrollo_2_2[columna_normalizada] = desarrollo_2_2["synopsis"].map(
        lambda texto: bool(patron.search(normalizar(texto)))
    )
    columnas_originales_desarrollo.append(columna_original)
    columnas_normalizadas_desarrollo.append(columna_normalizada)

desarrollo_2_2["marca_combinada_original"] = desarrollo_2_2[
    columnas_originales_desarrollo
].any(axis=1)
desarrollo_2_2["marca_combinada_normalizada"] = desarrollo_2_2[
    columnas_normalizadas_desarrollo
].any(axis=1)


def senales_fila(fila, sufijo):
    return ", ".join(
        nombre
        for nombre, columna, _ in patrones_regex
        if fila[f"{columna}_{sufijo}"]
    )


slugs_recuperados = [
    "espana-terra-incognita",
    "los-crimenes-del-invisible",
]
ejemplos_recuperados = desarrollo_2_2.loc[
    desarrollo_2_2["slug"].isin(slugs_recuperados)
].copy()
ejemplos_recuperados["fragmento_normalizado"] = ejemplos_recuperados["fragmento"].map(
    normalizar
)
ejemplos_recuperados["senales_originales"] = ejemplos_recuperados.apply(
    lambda fila: senales_fila(fila, "original"), axis=1
)
ejemplos_recuperados["senales_normalizadas"] = ejemplos_recuperados.apply(
    lambda fila: senales_fila(fila, "normalizada"), axis=1
)
print("Enlaces recuperados en desarrollo; no forman parte de las métricas:")
display(
    ejemplos_recuperados[
        [
            "slug",
            "title",
            "fragmento",
            "fragmento_normalizado",
            "senales_originales",
            "senales_normalizadas",
        ]
    ]
)

caso_homoglifos = desarrollo_2_2.loc[
    desarrollo_2_2["slug"].eq("mas-alla-13")
].copy()
caso_homoglifos["fragmento_normalizado"] = caso_homoglifos["fragmento"].map(
    normalizar
)
print("Homógrafos normalizados en un dominio desnudo, que la regex aprobada no marca:")
display(
    caso_homoglifos[
        [
            "slug",
            "title",
            "fragmento",
            "fragmento_normalizado",
            "marca_combinada_original",
            "marca_combinada_normalizada",
        ]
    ]
)

slugs_limpios_con_invisibles = ["hispania", "martin-lutero-ii"]
ejemplos_limpios_invisibles = desarrollo_2_2.loc[
    desarrollo_2_2["slug"].isin(slugs_limpios_con_invisibles)
].copy()
ejemplos_limpios_invisibles["caracteres_invisibles"] = ejemplos_limpios_invisibles[
    "synopsis"
].map(
    lambda texto: ", ".join(
        sorted({
            f"{unicodedata.name(caracter, 'SIN NOMBRE')} (U+{ord(caracter):04X})"
            for caracter in texto
            if unicodedata.category(caracter) == "Cf"
        })
    )
)
print("Sinopsis limpias de desarrollo con caracteres invisibles; no son contaminación por sí solos:")
display(
    ejemplos_limpios_invisibles[
        [
            "slug",
            "title",
            "caracteres_invisibles",
            "marca_combinada_original",
            "marca_combinada_normalizada",
        ]
    ]
)

Enlaces recuperados en desarrollo; no forman parte de las métricas:


,slug,title,fragmento,fragmento_normalizado,senales_originales,senales_normalizadas
280,espana-terra-incognita,"España, terra incognita",Link en nuestro sitio: h t t p s : / / lectorp...,link en nuestro sitio:https://lectorplus.invalid,,URL
283,los-crimenes-del-invisible,Los crímenes del invisible,Buscalo en w w w . libroplus . example,buscalo en www.libroplus.example,,URL


Homógrafos normalizados en un dominio desnudo, que la regex aprobada no marca:


,slug,title,fragmento,fragmento_normalizado,marca_combinada_original,marca_combinada_normalizada
263,mas-alla-13,Más Allá 13,Entrá a lеctorflаsh.test y leé sin límites.,entra a lectorflash.test y lee sin limites.,False,False


Sinopsis limpias de desarrollo con caracteres invisibles; no son contaminación por sí solos:


,slug,title,caracteres_invisibles,marca_combinada_original,marca_combinada_normalizada
428,hispania,Hispania,SOFT HYPHEN (U+00AD),False,False
132,martin-lutero-ii,Martín Lutero II,WORD JOINER (U+2060),False,False


La normalización elimina caracteres de formato (`Cf`) únicamente en la copia de trabajo. «Hispania» y «Martín Lutero II» muestran que puede haber caracteres invisibles en sinopsis limpias: por eso una marca requiere que alguna regex encuentre además una señal.

En «Más Allá 13» se convierten dos letras cirílicas visualmente similares a `e` y `a`, pero el texto contiene un dominio desnudo, sin `http(s)://` ni `www.`. Las regex del 2.1 siguen sin detectarlo; ampliarlas queda fuera de este apartado.